In [73]:
import torch
from torchvision import datasets, transforms


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)

In [74]:
from dropout import DropOut
import torch.nn as nn
from torch import Tensor

class NNDropOut(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(784, 32)
        self.relu1 = nn.ReLU()
        self.dropout1 = DropOut(p = 0.5)
        self.linear2 = nn.Linear(32, 16)

    def forward(self, x: Tensor) -> Tensor:
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.linear2(x)
        return x


class NNNormal(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(784, 128)
        self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

    def forward(self, x: Tensor) -> Tensor:
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu1(x)
        x = self.linear2(x)
        x = self.relu2(x)
        return x



In [75]:
import torch.optim as optim

model = NNDropOut()

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001) 

In [76]:
epoch = 1
for i in range(epoch):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        _, predicted = torch.max(y_pred.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

        if batch_idx % 100 == 99:
            avg_loss = running_loss / 100
            acc = 100 * correct / total
            print(f'Epoch [{i+1}/{epoch}], Batch [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {acc:.2f}%')
            running_loss = 0.0 

Epoch [1/1], Batch [100], Loss: 1.6532, Acc: 46.81%
Epoch [1/1], Batch [200], Loss: 1.0315, Acc: 56.50%
Epoch [1/1], Batch [300], Loss: 0.8803, Acc: 61.55%
Epoch [1/1], Batch [400], Loss: 0.8019, Acc: 64.66%
Epoch [1/1], Batch [500], Loss: 0.7625, Acc: 66.94%
Epoch [1/1], Batch [600], Loss: 0.6754, Acc: 68.65%
Epoch [1/1], Batch [700], Loss: 0.7042, Acc: 69.75%
Epoch [1/1], Batch [800], Loss: 0.6547, Acc: 70.79%
Epoch [1/1], Batch [900], Loss: 0.6391, Acc: 71.66%
Epoch [1/1], Batch [1000], Loss: 0.6440, Acc: 72.35%
Epoch [1/1], Batch [1100], Loss: 0.6243, Acc: 73.14%
Epoch [1/1], Batch [1200], Loss: 0.5982, Acc: 73.72%
Epoch [1/1], Batch [1300], Loss: 0.6361, Acc: 74.08%
Epoch [1/1], Batch [1400], Loss: 0.6010, Acc: 74.53%
Epoch [1/1], Batch [1500], Loss: 0.6058, Acc: 74.87%
Epoch [1/1], Batch [1600], Loss: 0.5832, Acc: 75.28%
Epoch [1/1], Batch [1700], Loss: 0.5950, Acc: 75.59%
Epoch [1/1], Batch [1800], Loss: 0.5785, Acc: 75.93%


In [71]:
# 1. Set model to evaluation mode
model.eval()

correct = 0
total = 0
test_loader = torch.utils.data.DataLoader(test_set, batch_size=32, shuffle=True)

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the network on the 10,000 test images: {accuracy:.2f}%')

# NNNormal: Accuracy of the network on the 10,000 test images: 96.76%
# NNDropOut: Accuracy of the network on the 10,000 test images: 97.49%
# Most impressive result was NNDropout with an initial hidden layer of 32 neurons and second hidden layer of 16 layers
# where the last training accuracy was areound 48% and the test time accuracy was 91%

Accuracy of the network on the 10,000 test images: 94.04%
